# What can the repository do, shown on real files?

Entry point of the notebooks. One cell per capability, in the order they
were built, every one running on files under `resources/`; the diary of
how they came about is [.docs/progress.md](../.docs/progress.md), the
index of all notebooks [README.md](README.md). Cell bodies live in
[ipynb-assets/nb000_capabilities.py](ipynb-assets/nb000_capabilities.py).

## Checks
- [x] BEDLAM frames, depth maps and person masks load; the hair part is in the mask
- [x] LiDAR simulation: Ouster presets 32–256 channels, placement, occlusion, noise, ego motion
- [x] SLOPER4D, LiDARHuman26M and Waymo load with LiDAR-to-mesh distances
- [x] A collated training batch scores a near-zero loss on its own labels
- [x] The fusion model runs forward with the TokenHMR ViT-H backbone
- [x] Sensor rigs of every dataset, mesh-LiDAR 3DPW records, Waymo pseudo-GT records
- [x] Scoreboard, showcase of the headline checkpoint, SAM 3D Body conversion, Waymo header

**Verdict:** every stage from BEDLAM frame to trained-model showcase runs on local data.

In [ ]:
import sys

sys.path.insert(0, "ipynb-assets")
from nb000_capabilities import *  # noqa: F403

## BEDLAM frames
Image, planar depth and the person boxes of five frames spread over the local groups.

In [ ]:
ctx = load_bedlam(frame_stride=5, n_sequences=5)

In [ ]:
show_frames(ctx)

The person mask is the union of `body`, `clothing` and `hair`; without the hair part the head lost its top and its returns.

In [ ]:
show_hair_mask(ctx, key=("20221024_3-10_100_batch01handhair_static_highSchoolGym", "seq_000099", "0075", "00"))

In [ ]:
show_surfaces_3d(ctx)

## Simulated LiDAR
One Ouster scan per frame on the image, the returns on each person against its dense surface, and the 32–256 channel grid from four placements.

In [ ]:
scan_overlay(ctx, preset="OS1-64")

In [ ]:
returns_vs_surface(ctx)

In [ ]:
resolution_grid(ctx, sample=0)

Interactive simulator (needs a live kernel): resolutions, placement, camera and LiDAR covers, object occlusion, jitter, outliers, channel dropout, miscalibration, ego speed.

In [ ]:
simulator_widget(ctx)

## Real datasets
One record each from SLOPER4D, LiDARHuman26M and Waymo; the 3D views colour the returns by distance to the labelled mesh.

In [ ]:
real = load_real(ctx, sloper_seq="seq008_running_001")

In [ ]:
real_points_vs_mesh(real)

## Training batch and model
Four collated records, the loss on their own labels, then the untrained fusion model with the TokenHMR ViT-H and its gate priors.

In [ ]:
batch = training_batch(real)

In [ ]:
out = model_forward(batch)

## Sensor rigs
Every dataset's calibration tree; tick up to three rigs to draw them together.

In [ ]:
sensor_rigs(max_rigs=3)

## Generated records
3DPW with LiDAR simulated on the clothing-offset mesh, and Waymo records with their pseudo-GT SMPL fit (re-roll picks new ones).

In [ ]:
mesh_lidar_3dpw(ctx.smpl, pattern="meshlidar/v1/threedpw_*.npz", n=4)

In [ ]:
pseudo_gt_waymo(ctx.smpl, labels="real/v1_pseudo3", n=3)

## Model, scoreboard and showcase
The block diagram, the main scoreboard group pulled from Helma, and the headline checkpoint against the baselines on the validation sets.

In [ ]:
architecture_figure(width=1400)

In [ ]:
scoreboard(pull=True, groups=(0,))

In [ ]:
show = run_showcase("resources/pretrained-checkpoints/ours/full-main-mixed-last.pt", "configs/full_main_mixed.yaml", "outputs/ours/full-main-mixed")

In [ ]:
showcase_figures(show)

In [ ]:
transfer_figure({"real-only": "full-real-only-000", "synth-only": "full-synth-only-000", "main-mixed (real + synth)": "full-main-mixed-000", "mix80 (real + 80 % synth)": "full-mix80-000"})

In [ ]:
distance_and_gates(show, "outputs/ours/full-main-mixed")

## Baseline conversion and header figure
SAM 3D Body's MHR mesh against the SMPL fitted to it, and a whole Waymo sweep with our meshes.

In [ ]:
sam3d_conversion(npz="outputs/baselines/sam3d-body/smoke/mhr_vs_smpl.npz")

In [ ]:
waymo_header(folder="outputs/figures/waymo_header", width=1400)